In [1]:
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'  # 必须在导入 Unsloth 之前设置！
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# 1. 加载基座模型（与 DAPT 一致）
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/root/autodl-tmp/DeepSeek-R1-Distill-Qwen-7B",
    max_seq_length=2048,
    load_in_4bit=True,
)

# 2. 加载 DAPT 适配器（40000 步最新版）
model.load_adapter("poetry-dapt-lora-adapter", adapter_name = "dapt")  # 确保路径正确

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/root/miniconda3/envs/agiclass/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:06<00:00, 52.31it/s]


/root/autodl-tmp/DeepSeek-R1-Distill-Qwen-7B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 470.09it/s]


In [3]:
# 3. 为 SFT 创建新的 LoRA 适配器
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
/root/miniconda3/envs/agiclass/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/root/miniconda3/envs/agiclass/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Unsloth 2026.4.6 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
# 4. 加载 SFT 数据集
dataset = load_dataset("json", data_files="poetry_sft_data.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.05, seed=42)

def format_instruction(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

train_dataset = dataset["train"].map(format_instruction, remove_columns=dataset["train"].column_names)
eval_dataset = dataset["test"].map(format_instruction, remove_columns=dataset["test"].column_names)

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(eval_dataset)}")

Generating train split: 1727 examples [00:00, 161380.48 examples/s]
Map: 100%|██████████| 87/87 [00:00<00:00, 4777.49 examples/s]

训练集样本数: 1640
验证集样本数: 87


In [6]:
sft_config = SFTConfig(
    output_dir="./poetry-sft-output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,               # SFT 学习率稍高
    max_seq_length=2048,
    dataset_text_field="text",
    packing=False,
    dataset_num_proc=4,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=30,
    save_strategy="steps",
    save_steps=30,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=True,
    max_steps=252,                    # 精准设置
    warmup_ratio=0.05,
    resume_from_checkpoint=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Tokenizing ["text"] (num_proc=4): 100%|██████████| 87/87 [00:01<00:00, 62.92 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [7]:
# 开始训练
trainer.train()

# 保存 SFT 适配器
model.save_pretrained("poetry-sft-adapter")
tokenizer.save_pretrained("poetry-sft-adapter")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,640 | Num Epochs = 3 | Total steps = 252
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
30,4.651682,3.499468
60,2.774952,2.543178
90,2.370480,2.274296
120,2.136079,2.161097
150,2.099180,2.086592
180,2.112473,2.027637
210,2.048654,1.986953
240,1.951810,1.959535
252,1.951810,1.956708


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/root/miniconda3/envs/agiclass/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/root/miniconda3/envs/agiclass/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE,

你是中国古典诗词赏析专家。<｜User｜>请解释“钟山风雨起苍黄”中“苍黄”的含义。<｜Assistant｜><think>
嗯，用户问的是“钟山风雨起苍黄”中“苍黄”的含义。首先，我得确定这是哪首诗。对了，这是毛泽东的《七律·人民解放军占领南京》。接下来，我需要分析“苍黄”在这里的具体含义。

“苍黄”通常指天空的颜色，但结合诗句语境，更可能是指天空的颜色变化。钟山指的是南京的钟山，那里有古塔，所以“钟山”在这里有古塔的意思。结合上下文，这句话描绘了南京古塔被雨风吹染成苍黄的颜色，显得非常凄凉。所以，“苍黄”在这里主要是指天空的颜色，结合诗句整体意境来理解。
</think>

“钟山风雨起苍黄”出自毛泽东的《七律·人民解放军占领南京》。此处“苍黄”指天空的颜色，结合诗句语境，指钟山古塔被风雨吹染成苍黄的颜色。
你是中国古典诗词赏析专家。<｜User｜>请将“战罢将军休复战，城中酒市已凄凉”翻译成现代汉语。<｜Assistant｜><think>
好的，我需要将诗句“战罢将军休复战，城中酒市已凄凉”翻译成现代汉语。首先，理解每一句的意思。第一句“战罢将军休复战”可以翻译为“战罢将军不再出战”，这里表达了战争结束，将军不再参战。第二句“城中酒市已凄凉”可以翻译为“城中酒市已变得凄凉”，描绘了战后酒市冷清的景象。整体上，诗句表达了战争结束后酒市冷清的景象。结合上下文，可以翻译为“战罢将军不再出战，城中酒市已变得凄凉。”
</think>

战罢将军不再出战，城中酒市已变得凄凉。


In [3]:
model.load_adapter("poetry-sft-adapter", adapter_name = "sft")

model.set_adapter(["dapt", "sft"])

/root/miniconda3/envs/agiclass/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Loading weights: 100%|██████████| 392/392 [00:00<00:00, 461.90it/s]
Qwen2ForCausalLM LOAD REPORT from: poetry-sft-adapter
Key                                                       | Status  | 
----------------------------------------------------------+---------+-
model.layers.{0...27}.self_attn.v_proj.lora_A.dapt.weight | MISSING | 
model.layers.{0...27}.mlp.gate_proj.lora_B.dapt.weight    | MISSING | 
model.layers.{0...27}.mlp.down_proj.lora_B.dapt.weight    | MISSING | 
model.layers.{0...27}.self_attn.q_proj.lora_B.dapt.weight | MISSING | 
model.layers.{0...27}.self_attn.k_proj.lora_B.dapt.weight | MISSING | 
model.layers.{0...27}.self_attn.o_proj.lora_A.dapt.weight | MISSING | 
model.layers.{0...27}.mlp.up_proj.lora

In [4]:
# 测试函数
def ask_poetry_expert(question):
    model_for_infer = FastLanguageModel.for_inference(model)
    messages = [
        {"role": "system", "content": "你是中国古典诗词赏析专家。"},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model_for_infer.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("助手")[-1].strip()

# 测试字词理解
# print(ask_poetry_expert("在诗句“美人清江畔，是夜越吟苦。”中，“美人”是什么意思？"))
# 测试诗句理解
print(ask_poetry_expert("西湖最盛，为春为月。一日之盛，为朝烟，为夕岚。
今岁春雪甚盛，梅花为寒所勒，与杏桃相次开发，尤为奇观。石篑数为余言:"
傅金吾园中梅，张功甫玉照堂故物也，急往观之。"余时为桃花所恋，竟不忍去。
湖上由断桥至苏堤一带，绿烟红雾，弥漫二十余里。歌吹为风，粉汗为雨，罗
纨之盛，多于堤畔之草，艳冶极矣。然杭人游湖，止午，未，申三时。其实湖光染翠之工，山岚设色之妙，皆在朝日始出，夕春未下，始极其浓媚。月景尤不可言，花态柳情，山容水意，别是一种趣味。此乐留与山僧游客受用，安可为俗士道哉? 这首词题为《晚游六桥待月记》，文中却始终没有正面描写待月的情景。请结
合材料二，简要分析作者这样处理的用意。"))

SyntaxError: unterminated string literal (detected at line 16) (1407994802.py, line 16)